# 03.04 — Typed Query Contracts

Orthograph provides a typed query contract for Cypher through `TypedCypherReadQueryModel` and
`TypedCypherWriteQueryModel`. These enforce at **class-definition time** that your Cypher is
syntactically valid and that every `$parameter` aligns with a declared `Params` field.

This notebook covers:
- Defining typed read and write queries (declarative style)
- How `cypher_template` is validated at definition time
- Running queries through a `CypherExecutor`
- Using `$param IS NULL OR` for optional filters
- The imperative escape hatch (and when you need it)

In [1]:
from typing import Any, Optional

from pydantic import BaseModel

from orthograph.definition import GraphDefinition, NodeModel
from orthograph.execution import CypherExecutor
from orthograph.queries import (
    CypherQueryDefinitionError,
    TypedCypherReadQueryModel,
    TypedCypherWriteQueryModel,
)

## 1. Define the domain model

We reuse the classic filmography domain: `Person`, `Movie`, and `ACTED_IN`.

In [2]:
from shared.filmography import ActedIn, Movie, Person

## 2. Declarative read query

Set a `cypher_template` ClassVar with `$param` placeholders that match your `Params` model
fields. The base class:
- Parses the Cypher at definition time (catches syntax errors immediately)
- Checks that every `$param` corresponds to a `Params` field
- Provides a default `build()` that returns `(cypher_template, params.model_dump())`

You only implement `materialize()` â€” the per-record mapping from raw driver records to
your `Output` model.

In [3]:
class MoviesByYearParams(BaseModel):
    released: int


class MoviesByYear(TypedCypherReadQueryModel[MoviesByYearParams, Movie]):
    """Find all movies released in a given year."""

    query_id = "movies_by_year"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) RETURN m.title, m.released, m.tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie(
            title=raw["m.title"],
            released=raw["m.released"],
            tagline=raw.get("m.tagline"),
        )


# The class was defined â€” that means cypher_template parsed and params aligned.
print("Query query_id:    ", MoviesByYear.query_id)
print("Backend:       ", MoviesByYear.backend)
print("Params fields: ", list(MoviesByYear.params_schema.model_fields.keys()))
print("Output fields: ", list(MoviesByYear.Output.model_fields.keys()))

Query name:     movies_by_year
Backend:        Backend.CYPHER
Params fields:  ['released']
Output fields:  ['title', 'released', 'year']


In [4]:
# Call build() -- pure, no session needed
query = MoviesByYear()
cypher, params = query.build(MoviesByYearParams(released=1999))
print("Cypher:", cypher)
print("Params:", params)

Cypher: MATCH (m:Movie {released: $released}) RETURN m.title, m.released, m.tagline
Params: {'released': 1999}


## 3. Declarative write query

Write queries work the same way, but implement `interpret_result()` instead of `materialize()`.  The method receives a `WriteResultSummary` (result counters), not a raw record.

In [5]:
class CreateMovieParams(BaseModel):
    title: str
    released: int


class CreateMovie(TypedCypherWriteQueryModel[CreateMovieParams, int]):
    """Create a new Movie node."""

    query_id = "create_movie"
    cypher_template = "CREATE (m:Movie {title: $title, released: $released}) RETURN m"

    def interpret_result(self, raw: Any) -> int:
        # In a real driver, raw is a WriteResultSummary with counters.
        # For now, we just return 1 to simulate "1 node created".
        return 1  # nodes created


query = CreateMovie()
cypher, params = query.build(CreateMovieParams(title="Speed", released=1994))
print("Cypher:", cypher)
print("Params:", params)

Cypher: CREATE (m:Movie {title: $title, released: $released}) RETURN m
Params: {'title': 'Speed', 'released': 1994}


## 4. Running queries through CypherExecutor

The `CypherExecutor` is the single I/O seam. It:
1. Validates params via `Params.model_validate()` (rejects bad types before any DB call)
2. Calls `build()` (pure â€” no session)
3. Parses the produced Cypher (runtime syntax check â€” critical for imperative queries)
4. Opens a session and executes
5. Materializes each record via `materialize()` (reads) or `interpret_result()` (writes)

Below we use a `FakeGraphSession` to demonstrate without a live database.

In [6]:
class FakeGraphSession:
    """Minimal stand-in for a graph driver session."""

    def __init__(self, records: list[dict[str, Any]]):
        self._records = records

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        pass

    def run(self, cypher: str, **params: Any):
        print(f"  [FakeSession] RUN: {cypher}")
        print(f"  [FakeSession] WITH: {params}")
        return self._records


# Simulate two movies returned from the database
fake_records = [
    {
        "m.title": "The Matrix",
        "m.released": 1999,
        "m.tagline": "Welcome to the Real World",
    },
    {"m.title": "Fight Club", "m.released": 1999, "m.tagline": None},
]

executor = CypherExecutor(lambda: FakeGraphSession(fake_records))
results = executor.read(MoviesByYear(), {"released": 1999})

print(f"\nReturned {len(results)} Movie objects:")
for movie in results:
    print(f"  {movie.title} ({movie.released})")

  [FakeSession] RUN: MATCH (m:Movie {released: $released}) RETURN m.title, m.released, m.tagline
  [FakeSession] WITH: {'released': 1999}

Returned 2 Movie objects:
  The Matrix (1999)
  Fight Club (1999)


## 5. Definition-time validation

If your `cypher_template` contains a `$param` that doesn't match a `Params` field, or
if the Cypher syntax is invalid, you get a `CypherQueryDefinitionError` **at class
definition time** â€” before any query ever runs. It derives from `CypherError`, the
base for every exception raised by the Cypher extension, so you can catch the whole
family with `except CypherError` or this specific subclass.

In [7]:
# Example 1: $param not declared on Params model
try:

    class BadParamQuery(TypedCypherReadQueryModel[MoviesByYearParams, Movie]):
        Params = MoviesByYearParams
        Output = Movie
        query_id = "bad_param"
        cypher_template = (
            "MATCH (m:Movie {year: $year}) RETURN m"  # $year not in Params!
        )

        def materialize(self, raw):
            return Movie(**raw)
except CypherQueryDefinitionError as e:
    print(f"Caught at definition time: {e}")

Caught at definition time: BadParamQuery: cypher_template uses parameter(s) ['$year'] not declared on MoviesByYearParams; MoviesByYearParams declares field(s) ['$released'] with no matching placeholder in cypher_template


In [8]:
# Example 2: Invalid Cypher syntax
try:

    class BadSyntaxQuery(TypedCypherReadQueryModel[MoviesByYearParams, Movie]):
        Params = MoviesByYearParams
        Output = Movie
        query_id = "bad_syntax"
        cypher_template = "MATSCH (m:Movie) RETRN m"  # typos!

        def materialize(self, raw):
            return Movie(**raw)
except CypherQueryDefinitionError as e:
    print(f"Caught at definition time: {e}")

Caught at definition time: BadSyntaxQuery: cypher does not parse: Expected program activity session close command or session close command. Line 1, Col: 7.
  MATSCH (m:Movie) RETRN m; MoviesByYearParams declares field(s) ['$released'] with no matching placeholder in cypher_template


## 6. Optional filters with `$param IS NULL OR`

For queries with optional parameters, use the Cypher pattern
`$param IS NULL OR n.prop = $param`. This keeps the query **static** (a single
`cypher_template` string), preserves definition-time validation, and lets you
pass `None` to skip a filter at runtime.

In [9]:
class MovieFilterParams(BaseModel):
    released: Optional[int] = None
    title: Optional[str] = None


class MoviesFiltered(TypedCypherReadQueryModel[MovieFilterParams, Movie]):
    """Find movies with optional year and title filters."""

    query_id = "movies_filtered"
    cypher_template = (
        "MATCH (m:Movie) "
        "WHERE ($released IS NULL OR m.released = $released) "
        "  AND ($title IS NULL OR m.title = $title) "
        "RETURN m.title, m.released"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie(
            title=raw["m.title"],
            released=raw["m.released"],
        )


# build() with partial params (title=None means "don't filter by title")
query = MoviesFiltered()
cypher, params = query.build(MovieFilterParams(released=1999))
print("Cypher:", cypher)
print("Params:", params)
print()

# build() with no filters at all
cypher, params = query.build(MovieFilterParams())
print("Cypher:", cypher)
print("Params:", params)

Cypher: MATCH (m:Movie) WHERE ($released IS NULL OR m.released = $released)   AND ($title IS NULL OR m.title = $title) RETURN m.title, m.released
Params: {'released': 1999, 'title': None}

Cypher: MATCH (m:Movie) WHERE ($released IS NULL OR m.released = $released)   AND ($title IS NULL OR m.title = $title) RETURN m.title, m.released
Params: {'released': None, 'title': None}


## 7. The imperative escape hatch

For the rare case where the query **shape** genuinely changes at runtime â€”
e.g. conditionally adding `OPTIONAL MATCH` clauses, choosing different relationship
types, or varying `RETURN` columns â€” you can override `build()` directly and skip
the `cypher_template`.

**Trade-offs:**
- No definition-time validation (syntax is only checked at runtime by the executor)
- A `UserWarning` is emitted at class-definition time to surface this
- Cannot be introspected by `validate_cypher()` or the catalogue's `describe()`

Prefer the `$param IS NULL OR` pattern whenever possible.

In [10]:
import warnings

from orthograph.cypher.bindings import CypherQueryData


class ActorFilmographyParams(BaseModel):
    name: str
    include_directed: bool = False


# This will emit a UserWarning â€” that's expected and intentional.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)

    class ActorFilmography(TypedCypherReadQueryModel[ActorFilmographyParams, Movie]):
        """Find movies for an actor, optionally including directed films.

        This query's shape changes based on include_directed â€” it adds
        an OPTIONAL MATCH clause conditionally. That structural change
        requires the imperative style.
        """

        query_id = "actor_filmography"

        def build(self, params: ActorFilmographyParams) -> CypherQueryData:
            q = "MATCH (p:Person {name: $name})-[:ACTED_IN]->(m:Movie) "
            if params.include_directed:
                q += "OPTIONAL MATCH (p)-[:DIRECTED]->(m) "
            q += "RETURN m.title, m.released"
            return q, {"name": params.name}

        def materialize(self, raw: dict[str, Any]) -> Movie:
            return Movie(title=raw["m.title"], released=raw["m.released"])


# Without directed
query = ActorFilmography()
cypher, params = query.build(
    ActorFilmographyParams(name="Keanu Reeves", include_directed=False)
)
print("Without directed:")
print("  Cypher:", cypher)
print("  Params:", params)
print()

# With directed
cypher, params = query.build(
    ActorFilmographyParams(name="Keanu Reeves", include_directed=True)
)
print("With directed:")
print("  Cypher:", cypher)
print("  Params:", params)

Without directed:
  Cypher: MATCH (p:Person {name: $name})-[:ACTED_IN]->(m:Movie) RETURN m.title, m.released
  Params: {'name': 'Keanu Reeves'}

With directed:
  Cypher: MATCH (p:Person {name: $name})-[:ACTED_IN]->(m:Movie) OPTIONAL MATCH (p)-[:DIRECTED]->(m) RETURN m.title, m.released
  Params: {'name': 'Keanu Reeves'}


## 8. Registering queries in a QueryCatalogue

A `QueryCatalogue` is a typed object registry. You register `ReadQueryModel` /
`WriteQueryModel` *instances*; it never does string-key dispatch and queries reference
their `Output` model by direct import. The catalogue introspects what it holds via
`describe()` and lists names via `names()` â€” both optionally filtered by backend.

The library ships this contract; the **application** owns registration (defines its
own queries and a composition root that registers them).


In [11]:
from orthograph.execution import Backend
from orthograph.queries import QueryCatalogue


query_catalogue = QueryCatalogue()
query_catalogue.register_read(MoviesByYear())
query_catalogue.register_write(CreateMovie())

for desc in query_catalogue.describe():
    print(f"{desc.query_id:16} kind={desc.kind:5} backend={desc.backend.value}")

movies_by_year   kind=read  backend=cypher
create_movie     kind=write backend=cypher


### Filtering by backend

`describe(backend=...)` and `names(backend=...)` return only the queries that target
a given backend. With no argument they return everything. This is the lightweight
hook for tooling that needs to enumerate, say, only the Cypher queries.


In [12]:
print("all names:    ", query_catalogue.names())
print("cypher names: ", query_catalogue.names(backend=Backend.CYPHER))
print("sql names:    ", query_catalogue.names(backend=Backend.SQLALCHEMY))

all names:     ['movies_by_year', 'create_movie']
cypher names:  ['movies_by_year', 'create_movie']
sql names:     []


## 9. Validating a catalogue against a graph model

`validate_catalogue(catalogue, model)` checks every registered query against a
`GraphDefinition` â€” **without a database**. Declarative Cypher queries are validated
via `validate_cypher` (unknown labels / rel types / properties become ERRORs).
Imperative or non-Cypher queries cannot be inspected statically and are reported as
`QUERY_UNVERIFIABLE` (INFO) â€” never silently skipped.


In [13]:
from orthograph.queries import validate_catalogue


graph_definition = GraphDefinition(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)

result = validate_catalogue(query_catalogue, graph_definition)
print("valid:", result.is_valid)
for issue in result.issues:
    print(" ", issue)

valid: False
  [ERROR] QUERY_UNKNOWN_PROPERTY: Query accesses property 'tagline' on Movie which is not in the model (entity=Movie.tagline)
  [ERROR] QUERY_RETURN_OUTPUT_MISMATCH: Query 'movies_by_year': Output is a NodeModel ('Movie') but RETURN contains no matching whole-node column (entity=movies_by_year)


A query that references something outside the model surfaces an ERROR. Here we
register a query against a `Studio` label the model does not declare.


In [14]:
class StudioParams(BaseModel):
    name: str


class MoviesByStudio(TypedCypherReadQueryModel[StudioParams, Movie]):
    query_id = "movies_by_studio"
    cypher_template = (
        "MATCH (s:Studio {name: $name})-[:MADE]->(m:Movie) RETURN m.title, m.released"
    )

    def materialize(self, raw: Any) -> Movie:
        return Movie(title=raw["m.title"], released=raw["m.released"])


bad_catalogue = QueryCatalogue()
bad_catalogue.register_read(MoviesByStudio())

bad_result = validate_catalogue(bad_catalogue, graph_definition)
print("valid:", bad_result.is_valid)
for issue in bad_result.errors:
    print(" ", issue)

valid: False
  [ERROR] QUERY_UNKNOWN_NODE_LABEL: Query references node label 'Studio' not in model (entity=Studio)
  [ERROR] QUERY_UNKNOWN_REL_TYPE: Query references relationship type 'MADE' not in model (entity=MADE)
  [ERROR] QUERY_INVALID_ENDPOINT: Query pattern (:Studio)-[:MADE]->(:Movie) does not match any declared 'MADE' relationship type (entity=MADE)
  [ERROR] QUERY_RETURN_OUTPUT_MISMATCH: Query 'movies_by_studio': Output is a NodeModel ('Movie') but RETURN contains no matching whole-node column (entity=movies_by_studio)


### Validating against a live database profile

`validate_catalogue_against_profile(catalogue, profile, model)` merges the query
check above with `compare(profile, model)` (the DB shape vs the model). You
pass a `GraphProfile` produced by a `GraphInspector` â€” *you* own the driver and run
`inspect()`, so the validator never touches a connection. See the Neo4j / Memgraph
notebooks for producing a real profile.


## 9b. Pagination with `PaginatedParams`

`PaginatedParams` is a `BaseModel` mixin that adds `skip` and `limit` to any `Params`
model. It is composable — inherit from it and add your own fields:

```python
from orthograph.query.pagination import PaginatedParams

class MoviesByYearParams(PaginatedParams):
    released: int
# Adds: skip (default 0, min 0) and limit (default 100, min 1, max 1000)
```

### The graphglot `LIMIT $param` gap (tech-debt E20/T7)

The bundled Cypher parser (graphglot) currently **rejects a parameterised `LIMIT $limit`**
— it parses `SKIP $skip` but not `LIMIT $limit`. Only a literal `LIMIT 100` is accepted.
This blocks pagination at two layers:

- **Definition time:** `_validate_declarative_cypher` raises `CypherQueryDefinitionError`
  on a `cypher_template` containing `LIMIT $limit`.
- **Runtime:** `CypherExecutor.read` re-parses the built Cypher via `_validate_cypher`,
  so an imperative `build()` that emits `LIMIT $limit` also raises `CypherSyntaxError`.

### Workaround: imperative `build()` with validated integer literals

The library's documented escape hatch is to omit `cypher_template` and override `build()`.
Since `PaginatedParams` validates `skip` and `limit` as bounded integers, it is safe to
inline them as integer literals — they cannot carry a Cypher fragment.

```python
class PagedMoviesByYear(TypedCypherReadQueryModel[PagedMoviesByYearParams, Movie]):
    name = "paged_movies_by_year"
    # No cypher_template — imperative because of the LIMIT $param graphglot gap.

    def build(self, params: PagedMoviesByYearParams) -> CypherQuery:
        # skip/limit are validated ints (skip>=0, 1<=limit<=1000).
        # Inlining as literals is injection-safe; $released stays a real parameter.
        cypher = (
            "MATCH (m:Movie {released: $released}) "
            "RETURN m.title AS title, m.released AS released "
            f"SKIP {int(params.skip)} LIMIT {int(params.limit)}"
        )
        return cypher, {"released": params.released}

    def materialize(self, raw: dict) -> Movie:
        return Movie.model_validate(raw)
```

The base emits a `UserWarning` for imperative queries (definition-time validation is
skipped); suppress it deliberately when the imperative choice is intentional.

This cell demonstrates the workaround and verifies the `PaginatedParams` field schema.


In [15]:
import warnings
from typing import Optional

from orthograph.cypher.bindings import CypherQueryData
from orthograph.queries import TypedCypherReadQueryModel
from orthograph.query.pagination import PaginatedParams


# --- Minimal domain model for demonstration ---
class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    released: int
    tagline: Optional[str] = None


class PagedMoviesByYearParams(PaginatedParams):
    """Year filter + skip/limit pagination."""

    released: int


# Suppress the UserWarning emitted for imperative queries (deliberate choice here).
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)

    class PagedMoviesByYear(TypedCypherReadQueryModel[PagedMoviesByYearParams, Movie]):
        """Paginated movie query — imperative build() because of graphglot LIMIT gap."""

        query_id = "paged_movies_by_year"

        def build(self, params: PagedMoviesByYearParams) -> CypherQueryData:
            cypher = (
                "MATCH (m:Movie {released: $released}) "
                "RETURN m.title AS title, m.released AS released, m.tagline AS tagline "
                f"SKIP {int(params.skip)} LIMIT {int(params.limit)}"
            )
            return cypher, {"released": params.released}

        def materialize(self, raw: dict) -> Movie:
            return Movie.model_validate(raw)


# Verify the params schema exposes skip/limit as documented fields.
schema = PagedMoviesByYearParams.model_json_schema()
props = list(schema["properties"].keys())
print("PagedMoviesByYearParams fields:", props)
assert "skip" in props and "limit" in props and "released" in props

# Verify build() produces valid Cypher (the literal-inlining approach).
q = PagedMoviesByYear()
cypher, qparams = q.build(PagedMoviesByYearParams(released=1999, skip=0, limit=10))
print("Built Cypher:", cypher)
print("Driver params (released only):", qparams)
assert "$skip" not in cypher and "$limit" not in cypher, "skip/limit must be literals"
assert "$released" in cypher
assert qparams == {"released": 1999}

# Verify PaginatedParams enforces bounds.
try:
    PagedMoviesByYearParams(released=1999, skip=-1, limit=10)
    raise AssertionError("should have rejected skip=-1")
except Exception as e:
    print("skip=-1 rejected:", type(e).__name__)

try:
    PagedMoviesByYearParams(released=1999, skip=0, limit=0)
    raise AssertionError("should have rejected limit=0")
except Exception as e:
    print("limit=0  rejected:", type(e).__name__)

print("PaginatedParams bounds enforcement: OK")

PagedMoviesByYearParams fields: ['skip', 'limit', 'released']
Built Cypher: MATCH (m:Movie {released: $released}) RETURN m.title AS title, m.released AS released, m.tagline AS tagline SKIP 0 LIMIT 10
Driver params (released only): {'released': 1999}
skip=-1 rejected: ValidationError
limit=0  rejected: ValidationError
PaginatedParams bounds enforcement: OK


## 10. Summary

| Style | Set `cypher_template`? | Override `build()`? | Definition-time validation? | When to use                       |
|-------|:---------------------:|:-------------------:|:---------------------------:|-----------------------------------|
| **Declarative** | Yes | No (free default) | Yes | mayority of queries â€” fixed shape |
| **Imperative** | No | Yes | No (runtime only) | Structural shape changes          |

**Guidelines:**
- Use `$param IS NULL OR n.prop = $param` for optional filters â€” keeps queries declarative.
- Use the imperative escape hatch only when the query *topology* changes (conditional
  `MATCH`/`OPTIONAL MATCH`, dynamic relationship types, variable `RETURN` columns).
- The executor always runs `parse_cypher()` on the produced string, so even imperative
  queries get a syntax check before hitting the database.